# Phase V: Model Explainability & Causal Pathway Visualization
### Applying Explainable AI (SHAP) & Interactive Network Reconstruction in TNBC

In **Phase III & IV**, we reconstructed the triangulated causal network (`TNBC_Causal_Tripartite_GRN.csv`) and prioritized key transcriptional master drivers (`Top_TNBC_Causal_Drivers.csv`).

Now in **Phase V**, we execute **Work Package 6** from the thesis proposal:
1. **Explainable AI (SHAP Feature Attribution)**: Quantify the non-linear causal impact of upstream enhancer perturbations and master transcription factors on key TNBC oncogenes (e.g., *MYC*, *TP53*, *EGFR*).
2. **Feature Importance & Directionality**: Interpret positive and negative regulatory mechanisms using SHAP values.
3. **Interactive Graph Visualization (PyVis & Cytoscape)**: Export interactive, navigable network graphs and Cytoscape-compatible edge/node tables for the master TNBC regulatory core.
4. **Export Publication Deliverables**: Save `Figure_6_Explainability_and_Causal_Pathways.png/.tiff` and attribution tables.

### 1. Environment & Path Setup

In [1]:
import os
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
import networkx as nx
from pyvis.network import Network
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

BASE_DIR = r'O:\Thesis\TNBC_Work\TNBC_Final_Dataset'
if not os.path.exists(BASE_DIR):
    BASE_DIR = os.getcwd()

GRN_FILE = os.path.join(BASE_DIR, 'TNBC_Causal_Tripartite_GRN.csv')
DRIVERS_FILE = os.path.join(BASE_DIR, 'Top_TNBC_Causal_Drivers.csv')
FIG_DIR = os.path.join(BASE_DIR, 'manuscript_figures')
TAB_DIR = os.path.join(BASE_DIR, 'manuscript_tables')
OUT_HTML = os.path.join(BASE_DIR, 'TNBC_Causal_Network_Interactive.html')
OUT_SHAP_CSV = os.path.join(TAB_DIR, 'Table_5_SHAP_Perturbation_Attribution.xlsx')

print(f'Working directory: {BASE_DIR}')
print(f'Causal GRN: {GRN_FILE} (Exists: {os.path.exists(GRN_FILE)})')
print(f'Drivers list: {DRIVERS_FILE} (Exists: {os.path.exists(DRIVERS_FILE)})')

Working directory: O:\Thesis\TNBC_Work\TNBC_Final_Dataset
Causal GRN: O:\Thesis\TNBC_Work\TNBC_Final_Dataset\TNBC_Causal_Tripartite_GRN.csv (Exists: True)
Drivers list: O:\Thesis\TNBC_Work\TNBC_Final_Dataset\Top_TNBC_Causal_Drivers.csv (Exists: True)


### 2. Prepare Causal Feature Matrix & Train Explainable Predictive Model
We model how upstream master regulatory transcription factors and perturbed enhancers control downstream target gene expression in TNBC.

In [2]:
print('Loading Causal Network sample for Explainable AI...')
# Load representative causal paths across top drivers
df_grn = pd.read_csv(GRN_FILE, nrows=250000)
df_drivers = pd.read_csv(DRIVERS_FILE)

top_tfs = df_drivers.head(10)['tf_name'].tolist()
grn_top = df_grn[df_grn['tf_name'].isin(top_tfs)].copy()
print(f'Filtered to {len(grn_top):,} paths regulated by top 10 master drivers.')

# Pivot to construct TF -> Gene causal feature matrix
pivot_matrix = grn_top.pivot_table(index='target_gene', columns='tf_name', values='causal_weight', fill_value=0.0)

# Target variable: Target gene responsiveness (mean absolute log2 fold change)
gene_responsiveness = grn_top.groupby('target_gene')['log2_fc'].apply(lambda x: np.mean(np.abs(x)))
y = gene_responsiveness.loc[pivot_matrix.index].values
X = pivot_matrix.values
feature_names = list(pivot_matrix.columns)

print(f'Constructed feature matrix: {X.shape[0]} target genes x {X.shape[1]} master regulators.')

# Train Gradient Boosted Decision Trees (XGBoost)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

r2_score = model.score(X_test, y_test)
print(f'Trained XGBoost Causal Response Model | Test R2: {r2_score:.3f}')

Loading Causal Network sample for Explainable AI...
Filtered to 38,545 paths regulated by top 10 master drivers.


Constructed feature matrix: 464 target genes x 10 master regulators.
Trained XGBoost Causal Response Model | Test R2: 0.320


### 3. Compute SHAP Values (TreeExplainer)
We compute exact Shapley values to interpret the non-linear contribution of each master regulator to downstream gene expression regulation.

In [3]:
print('Computing SHAP values using TreeExplainer...')
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# Summary of feature attributions
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
df_shap = pd.DataFrame({
    'Master_Regulator': feature_names,
    'Mean_Absolute_SHAP_Impact': mean_abs_shap,
    'Feature_Weight_Sum': X.sum(axis=0)
}).sort_values(by='Mean_Absolute_SHAP_Impact', ascending=False).reset_index(drop=True)

print('\n--- Master Regulators Ranked by SHAP Explanatory Power ---')
display(df_shap)

# Export Table 5
df_shap.to_excel(OUT_SHAP_CSV, index=False)
print(f'Saved SHAP Attribution Table: {OUT_SHAP_CSV}')

Computing SHAP values using TreeExplainer...

--- Master Regulators Ranked by SHAP Explanatory Power ---


,Master_Regulator,Mean_Absolute_SHAP_Impact,Feature_Weight_Sum
0,JUND,0.029582,3798.178014
1,MYC,0.017427,3391.990606
2,MAX,0.012536,3594.961088
3,EP300,0.011668,3690.947098
4,CTCF,0.008296,4484.269562
5,HDAC2,0.006833,3451.964606
6,SIN3A,0.006764,3713.954473
7,SP1,0.005498,3979.863353
8,POLR2A,0.004045,4186.990148
9,FOS,0.003714,3931.076751


Saved SHAP Attribution Table: O:\Thesis\TNBC_Work\TNBC_Final_Dataset\manuscript_tables\Table_5_SHAP_Perturbation_Attribution.xlsx


### 4. Interactive Causal Network Visualization (PyVis & Cytoscape)
We generate an interactive, draggable network showing the directed flow from Master TFs $\rightarrow$ Regulatory Enhancers $\rightarrow$ Downstream Targets, exported to standalone HTML.

In [4]:
print('Building interactive tripartite subnetwork for top TNBC drivers...')

# Select core subnetwork (Top 5 TFs, Top 15 Enhancers, Top 30 Target Genes)
core_tfs = df_drivers.head(5)['tf_name'].tolist()
sub_grn = df_grn[df_grn['tf_name'].isin(core_tfs)].sort_values(by='causal_weight', ascending=False).head(120)

net = Network(height='700px', width='100%', directed=True, bgcolor='#ffffff', font_color='#222222')
net.force_atlas_2based(gravity=-60, central_gravity=0.005, spring_length=120, spring_strength=0.08)

# Color schemes for tripartite layers
# TF: Coral / Red | Enhancer: Purple / Mauve | Target Gene: Teal / Blue
added_nodes = set()

for _, row in sub_grn.iterrows():
    tf = row['tf_name']
    enh = row['enhancer_region']
    gene = row['target_gene']
    weight = float(row['causal_weight'])
    direction = row['effect_direction']
    
    # Add TF node
    if tf not in added_nodes:
        net.add_node(tf, label=tf, color='#b2182b', size=25, title=f'Master TF: {tf}', shape='diamond')
        added_nodes.add(tf)
    
    # Add Enhancer node
    if enh not in added_nodes:
        short_enh = enh.split(':')[0] + ':' + enh.split(':')[1][:5] + '...'
        net.add_node(enh, label=short_enh, color='#7570b3', size=15, title=f'Enhancer Locus: {enh}', shape='dot')
        added_nodes.add(enh)
        
    # Add Target Gene node
    if gene not in added_nodes:
        net.add_node(gene, label=gene, color='#1b9e77', size=18, title=f'Target Gene: {gene}', shape='ellipse')
        added_nodes.add(gene)
    
    # Edges: TF -> Enhancer (Physical Binding)
    net.add_edge(tf, enh, color='#7570b3', width=1.5, title='ENCODE Physical Binding')
    
    # Edges: Enhancer -> Target Gene (Causal Perturbation)
    edge_col = '#d95f02' if direction == 'upregulated' else '#2b5c8f'
    net.add_edge(enh, gene, color=edge_col, width=max(1.0, min(weight/8.0, 4.5)),
                 title=f'Causal Impact: {direction} (weight: {weight:.2f})')

net.write_html(OUT_HTML)
print(f'Exported Interactive Network Graph to: {OUT_HTML}')

# Export Cytoscape tables for publication and external exploration
cyto_edges_path = os.path.join(TAB_DIR, 'Cytoscape_Network_Edges.csv')
sub_grn.to_csv(cyto_edges_path, index=False)
print(f'Exported Cytoscape Edge Table: {cyto_edges_path}')

Building interactive tripartite subnetwork for top TNBC drivers...
Exported Interactive Network Graph to: O:\Thesis\TNBC_Work\TNBC_Final_Dataset\TNBC_Causal_Network_Interactive.html
Exported Cytoscape Edge Table: O:\Thesis\TNBC_Work\TNBC_Final_Dataset\manuscript_tables\Cytoscape_Network_Edges.csv


### 5. Export Phase V Publication Figure (Figure 6)

In [5]:
print('Rendering Figure 6: Model Explainability & Causal Attribution (Nature standard)...')

plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.titlesize'] = 0
plt.rcParams['axes.edgecolor'] = '#222222'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['axes.labelsize'] = 9

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6))

# (a) SHAP Feature Importance of Master Regulators
sns.barplot(data=df_shap, x='Mean_Absolute_SHAP_Impact', y='Master_Regulator', hue='Master_Regulator', legend=False,
            ax=axes[0], palette='flare', edgecolor='#222222', linewidth=0.6)
axes[0].set_xlabel('Mean |SHAP Value| (Causal Attribution)')
axes[0].set_ylabel('Master Transcriptional Driver')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].text(-0.16, 1.06, 'a', transform=axes[0].transAxes, fontsize=11, fontweight='bold')

# (b) SHAP Value Distribution vs Feature Weight (Top Driver: CTCF/POLR2A)
top_feature_idx = 0
top_feature_name = df_shap.iloc[0]['Master_Regulator']
axes[1].scatter(X[:, top_feature_idx], shap_values[:, top_feature_idx], color='#b2182b', s=8, alpha=0.5, rasterized=True)
axes[1].axhline(0, color='#666666', linestyle='--', linewidth=0.7)
axes[1].set_xlabel(f'Causal Regulatory Weight ({top_feature_name})')
axes[1].set_ylabel('SHAP Value (Impact on Regulation)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].text(-0.14, 1.06, 'b', transform=axes[1].transAxes, fontsize=11, fontweight='bold')

plt.tight_layout()
fig6_png = os.path.join(FIG_DIR, 'Figure_6_Explainability_and_Causal_Pathways.png')
fig6_tiff = os.path.join(FIG_DIR, 'Figure_6_Explainability_and_Causal_Pathways.tiff')
fig.savefig(fig6_png, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(fig6_tiff, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Exported Figure 6: {fig6_png} & {fig6_tiff}')

print('\nPHASE V COMPLETE! Model explainability and interactive causal network visualization finished.')

Rendering Figure 6: Model Explainability & Causal Attribution (Nature standard)...


Exported Figure 6: O:\Thesis\TNBC_Work\TNBC_Final_Dataset\manuscript_figures\Figure_6_Explainability_and_Causal_Pathways.png & O:\Thesis\TNBC_Work\TNBC_Final_Dataset\manuscript_figures\Figure_6_Explainability_and_Causal_Pathways.tiff

PHASE V COMPLETE! Model explainability and interactive causal network visualization finished.
